# Model D on Colab — fine-tune Laya on a GPU
The Mac smoke test projected ~8.6 h per epoch (stop rule: > 3 h → Colab). This notebook runs **the same `train_arm_d.py`** on a Colab GPU. Training stays fp32; only the device changes.

**Runtime → Change runtime type → GPU** (L4 or A100 if available; T4 works but is slower).

**Google Drive layout** — mirror the repo, so the script's relative paths work unchanged:
```
MyDrive/laya-arm-d/
├── laya-discover/train_arm_d.py
└── topic-classification/
    ├── data/aa_dataset-tickets-multi-lang-5-2-50-version.csv
    └── artifacts/splits.csv
```
Checkpoints (~0.85 GB per epoch, ~3.4 GB per seed) and predictions are written back to Drive, so a disconnect loses at most the epoch in progress.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/laya-arm-d"
import os
for f in ["laya-discover/train_arm_d.py",
          "topic-classification/data/aa_dataset-tickets-multi-lang-5-2-50-version.csv",
          "topic-classification/artifacts/splits.csv"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"missing on Drive: {f}"
os.makedirs(f"{ROOT}/laya-discover/logs", exist_ok=True)
print("Drive layout ✓")

In [ ]:
# Same library versions as the Mac run (requirements.txt); Colab's own CUDA torch is kept
!pip -q install laya==0.3.28 transformers==5.18.0 pandas==3.0.6 scikit-learn==1.9.1
import torch, transformers, laya
print("torch", torch.__version__, "· transformers", transformers.__version__, "· laya", laya.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Smoke test — 200 tickets, 1 epoch (~minutes)
Gate: loss finite, epoch saves and reloads, validation scores. Read the epoch time: × 58 ≈ one full epoch (11,670 / 200).

In [ ]:
%cd {ROOT}/laya-discover
!python train_arm_d.py --seed 42 --epochs 1 --limit 200 --val-limit 100 --tag colab_smoke 2>&1 | grep -E "^\[|Error|ABORT" | tee logs/arm_d_colab_smoke_seed42.log

## Full run — one seed per run
Change `SEED` to 43 and 44 for the other two runs. Each epoch is saved to `checkpoints/seed{SEED}/epoch{k}` and scored on validation as it finishes.

In [ ]:
SEED = 42
%cd {ROOT}/laya-discover
!python train_arm_d.py --seed {SEED} 2>&1 | grep --line-buffered -E "^\[|Error|ABORT" | tee logs/arm_d_seed{SEED}.log